# 05 · Database and persistence

The retrieval artefacts are now split across three persistent stores:

| Dir | Holds |
|---|---|
| `database/` | SQLite metadata: images, gallery records, retrieval logs |
| `embeddings/` | npz cache of full-dataset embeddings per modality |
| `faiss/` | persisted FAISS gallery files (`.index` + `_meta.json`) |

The cache key is a **config hash** (dataset + model config + checkpoint mtime), so
any change invalidates the cache automatically.

In [ ]:
import os, sys
PROJ = os.path.dirname(os.getcwd()) if os.path.split(os.getcwd())[1] == 'notebooks' else os.getcwd()
if PROJ not in sys.path: sys.path.insert(0, PROJ)
nb_dir = os.path.join(PROJ, 'notebooks')
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import utils
print('project root:', PROJ)


In [ ]:
import os, time, glob
for d in ('embeddings', 'faiss', 'database'):
    files = sorted(glob.glob(os.path.join(d, '*')))
    files = [os.path.basename(f) for f in files if not f.endswith('README.md')]
    print(f'{d}/  ({len(files)} generated files)')
    for f in files[:4]: print('   ', f)
    if len(files) > 4: print(f'    … {len(files)-4} more')

In [ ]:
# 1) Config-hash stability + invalidation.
from src.database import compute_cache_key
from src.utils.config import load_config
cfg = load_config('configs/default.yaml')   # merges defaults (adds outputs.model_dir)
ckpt = os.path.join(cfg['outputs']['model_dir'], 'best_model', 'model.pt')
h1 = compute_cache_key(cfg, ckpt)
h2 = compute_cache_key(cfg, ckpt)
cfg2 = dict(cfg); cfg2['model'] = dict(cfg['model']); cfg2['model']['embedding_dim'] = 256
h3 = compute_cache_key(cfg2, ckpt)
print('same config -> same hash:', h1 == h2, f'({h1})')
print('model change -> new hash:', h1 != h3, f'({h3})')

In [ ]:
# 2) EmbeddingStore: cold compute vs warm load.
from src.database import EmbeddingStore
store = EmbeddingStore('embeddings')
t0 = time.perf_counter(); cached = store.load('optical', h1); t_warm = time.perf_counter()-t0
print(f'warm load  : {t_warm*1000:.1f} ms  (embeddings {cached[0].shape} from disk)')
# Full recompute of 6000 embeddings through the network takes ~60s on CPU;
# the pipeline only pays that once per (config, checkpoint) pair.

In [ ]:
# 3) IndexStore: reload a gallery and search it.
import numpy as np
from src.database import IndexStore
istore = IndexStore('faiss')
P = utils.load_pipeline('configs/default.yaml')
n = len(P['full_ds']); gallery_ids = np.arange(n//2, n)
g = istore.load('optical', P['engine'].config_hash, gallery_ids)
print('reloaded gallery:', g.size, 'vectors, dim', g.index.dim)
scores, ids = g.index.search(P['engine'].cache_full_embeddings('optical')[0][:1], 5)
print('search on reloaded index -> top score', round(float(scores[0,0]), 3))

In [ ]:
# 4) MetadataStore: images, galleries, retrieval log.
from src.database import MetadataStore
db = MetadataStore('database/metadata.db')
print('db stats     :', db.stats())
print('class counts :', db.class_counts())
print('galleries    :', [(g['name'], g['num_vectors']) for g in db.list_galleries()])
print('recent logs  :', len(db.recent_retrievals(10)))

### Warm-start timing summary
On a cold start the pipeline embeds every gallery image through the network and
builds each index (~minutes). On a warm start it reloads the cached galleries from
`faiss/` in well under a second. The web app uses exactly this path: restart
`uvicorn api.app:app` and watch the startup log say `loading cached gallery …`.

In [ ]:
print('stores ready. You can now `uvicorn api.app:app` for the web UI, or')
print('browse the other notebooks to explore retrieval & index flavours.')